# Introduction to Hugging Face and Generative AI

In this notebook, we explored the fundamentals of **Generative AI** using **Hugging Face** pre-trained models. We learned how to access and use AI models without training them from scratch.

## Hugging Face

Hugging Face is a popular platform that hosts thousands of pre-trained AI models for tasks such as text generation, image generation, text-to-speech, translation, summarization, and more. Instead of building and training our own models, we can directly download and use these models for inference.

## Hugging Face Access Token

Some models require authentication before they can be accessed. We created a **Hugging Face Access Token** and stored it securely in the **Google Colab Secrets** section as `HF_TOKEN`. This allows the notebook to access Hugging Face models securely without exposing the token in the code.

## Loading Pre-trained Models

The notebook downloads the required pre-trained models from Hugging Face and loads them into memory. Once loaded, the models are ready to perform different AI tasks such as image generation and speech generation.

## Using a T4 GPU

A **T4 GPU** was used to efficiently run the AI models and perform faster inference.

## Text-to-Image Generation

We used the **Stable Diffusion (SDXL)** model to generate images from text prompts. The model interprets the user's description and creates a corresponding image.

**Workflow:**

**Text Prompt → Stable Diffusion Model → Generated Image**

## Text-to-Speech Generation

We also used the **SpeechT5** model to convert text into natural-sounding speech by using pre-trained voice embeddings.

**Workflow:**

**Input Text → SpeechT5 Model → Generated Audio**

## Summary

- Learned about Hugging Face and its pre-trained AI models.
- Created and securely stored a Hugging Face Access Token in Google Colab Secrets.
- Downloaded and used pre-trained models from Hugging Face.
- Used a T4 GPU for model inference.
- Generated images from text using Stable Diffusion.
- Generated speech from text using SpeechT5.

In [ ]:
!pip install -q --upgrade transformers==4.56.2 diffusers==0.32.2

In [ ]:
from huggingface_hub import login
from google.colab import userdata


hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)


In [ ]:

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from IPython.display import display
from diffusers import DiffusionPipeline
import torch

pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
pipe.to("cuda")

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = pipe(prompt=prompt, num_inference_steps=30).images[0]

display(image)


In [ ]:

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from diffusers import DiffusionPipeline
import torch

base = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
base.to("cuda")
refiner = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-refiner-1.0", text_encoder_2=base.text_encoder_2, vae=base.vae, torch_dtype=torch.float16, use_safetensors=True, variant="fp16",)
refiner.to("cuda")

# Define how many steps and what % of steps to be run on each experts (80/20) here
n_steps = 40
high_noise_frac = 0.8

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

# run both experts
image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images

image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]

display(image)

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])